# 4. Next release: a relational database (demonstration)

*Minicourse: Using and contributing to the LaStBeRu Strong Lensing compilation*

This demonstration runs on `che.cbpf.br`, where PostgreSQL listens only on `localhost`.

## Why change?

As seen in notebook 1, `Database.csv` has one line per **system**, with the records of different papers flattened with ` § `:

```
JNAME                   J114833.1+193003.2
Reference               Diehl et al. (2009) § Moustakas et al. (2012) § Stark et al. (2013)
Original_ID             SDSS J114833.14+193003.2 § SDSSJ114833.14+193003.2 § 1.0
z_L                     0.4457 § 0.4457 § 0.444
```

Problems with this format:

* a row of a source table has no identifier of its own; if two systems are merged or split, its history is lost;
* the `JNAME` comes from one record's coordinates, so it changes when that record changes;
* corrections (a coordinate fixed by hand, a BCG chosen after visual inspection) have nowhere to be recorded;
* every query starts by splitting strings.

## The model

Each row of a processed table is a **record**, identified by (`source`, `record_id`). Records are grouped into **systems** through a link table, so regrouping does not touch the records:

```mermaid
erDiagram
    sources ||--o{ records : publishes
    records ||--o| system_records : "belongs to"
    systems ||--|{ system_records : groups
    systems |o--o| records : "anchored on"
    systems ||--o{ jname_v1_map : "old names"
    records ||..o| manual_reviews : "reviewed by"

    sources {
        text source PK
        text reference
        text source_kind "literature | cluster_catalog | compilation"
    }
    records {
        text source PK,FK
        int record_id PK
        text original_id
        float ra
        float dec
        float sigma_arcsec "positional uncertainty"
        text role "lens | image_arc | parent_sample | cluster_catalog"
        text scale "galaxy | group | cluster"
        jsonb raw "full row of the processed CSV"
    }
    systems {
        bigint system_id PK
        text jname UK
        float ra
        float dec
        text coordinate_origin "tractor | literature | bcg | manual"
        text anchor_source FK
        int anchor_record_id FK
        bigint tractor_ls_id
        float offset_arcsec
        text review_flag
        text anchor_note
    }
    system_records {
        bigint system_id FK
        text source PK
        int record_id PK
        text role
    }
    manual_reviews {
        text source PK
        int record_id PK
        jsonb decision
        text note
    }
    jname_v1_map {
        text jname_v1 PK
        bigint system_id PK,FK
        text method
        float separation_arcsec
    }
```

* `records.raw` keeps the full row of the processed CSV as JSON, including non-standard columns; the original files stay in `dataset/`.
* `systems` stores the adopted center and **how** it was chosen: a Legacy Surveys DR10 Tractor source, the literature position, or the brightest cluster galaxy. The record that supports the choice is in `anchor_source`, `anchor_record_id`.
* `system_id` is regenerated at every load and must not be stored outside the database. Human decisions (`manual_reviews`) are therefore keyed by the record (`source`, `record_id`), never by `system_id` or `JNAME`, and survive reloads as long as each extraction keeps its `record_id` numbering.
* `jname_v1_map` links each name of the public release to the new systems.

Schema: `slcomp-dev/02_Data/02_Database/schema.sql`; loader: `load_postgres.py`.

In [1]:
import os

import pandas as pd
import psycopg2

DSN = os.environ.get("SLCOMP_DSN", "host=localhost dbname=slcomp")
connection = psycopg2.connect(DSN)


def sql(query, params=None):
    """Run a query in the slcomp schema and return a DataFrame."""
    with connection.cursor() as cursor:
        cursor.execute("SET search_path TO slcomp")
        cursor.execute(query, params)
        return pd.DataFrame(
            cursor.fetchall(), columns=[d.name for d in cursor.description]
        )


sql("""
SELECT (SELECT count(*) FROM sources) AS sources,
       (SELECT count(*) FROM records) AS records,
       (SELECT count(*) FROM systems) AS systems,
       (SELECT count(*) FROM manual_reviews) AS manual_reviews
""")

,sources,records,systems,manual_reviews
0,75,44952,31137,581


The new database has fewer systems than the 31,569 lines of `Database.csv`: it joins records that the historical merge kept apart (mostly cluster catalogs and arcs with their parent cluster) and splits others (catalogs that point to different galaxies of the same field).

## The Cosmic Horseshoe, again

The three records of notebook 1, now linked to one system:

In [2]:
sql("""
SELECT s.system_id, s.jname, r.source, r.record_id, r.original_id, r.ra, r.dec, r.scale,
       r.raw->>'Spec-z_L' AS z_l, r.raw->>'Spec-z_S' AS z_s, r.theta_e, r.theta_e_method
FROM systems AS s
JOIN system_records AS sr USING (system_id)
JOIN records AS r USING (source, record_id)
WHERE s.jname LIKE 'J114833%%'
""")

,system_id,jname,source,record_id,original_id,ra,dec,scale,z_l,z_s,theta_e,theta_e_method
0,2087,J114833.13+193003.1,Diehl_et_al._2009,3,SDSS J114833.14+193003.2,177.138083,19.500889,galaxy,0.4457,2.3811,NaN,NaN
1,2087,J114833.13+193003.1,Moustakas_et_al._2012,6327,SDSSJ114833.14+193003.2,177.138083,19.500889,group,0.4457,2.3811,5.1,SIE model
2,2087,J114833.13+193003.1,Stark_et_al._2013,1,1.0,177.138074,19.500873,galaxy,0.444,2.379,NaN,NaN


Three details connect to notebook 1:

* `theta_e` holds only Einstein radii from lens modelling. The value of Diehl et al. (2009), a fit of a circle to the arc, stays in `raw`.
* `original_id = 1.0` for Stark et al. (2013) is the extraction error seen in notebook 1; the record keeps it until the processed table is corrected.
* `scale` differs between records (`galaxy`, `group`): each record keeps what its source says.

The center of the system is the Tractor source found with the TAP query in notebook 1 (`ls_id = 10995579072021591`):

In [3]:
sql("""
SELECT s.system_id, s.jname, s.ra, s.dec, s.coordinate_origin, s.tractor_ls_id,
       s.anchor_source, s.anchor_record_id, round(s.offset_arcsec::numeric, 3) AS offset_arcsec,
       m.jname_v1
FROM systems AS s
LEFT JOIN jname_v1_map AS m USING (system_id)
WHERE s.jname LIKE 'J114833%%'
""")

,system_id,jname,ra,dec,coordinate_origin,tractor_ls_id,anchor_source,anchor_record_id,offset_arcsec,jname_v1
0,2087,J114833.13+193003.1,177.13807,19.500877,tractor,10995579072021591,Diehl_et_al._2009,3,0.060,J114833.1+193003.2


The new `JNAME` is built from this center and truncated (RA to 0.01 s, Dec to 0.1″), so `J114833.13+193003.1` replaces `J114833.1+193003.2`; the old name stays in `jname_v1_map`.

## Questions that are now one query

**How many sources report each system?**

In [4]:
sql("""
SELECT n_sources, count(*) AS systems
FROM (SELECT system_id, count(DISTINCT source) AS n_sources FROM system_records GROUP BY system_id) AS t
GROUP BY n_sources ORDER BY n_sources
""")

,n_sources,systems
0,1,23120
1,2,6525
2,3,884
3,4,345
4,5,203
5,6,36
6,7,15
7,8,4
8,9,2
9,10,2


**Where do the system centers come from?**

In [5]:
sql("SELECT coordinate_origin, count(*) FROM systems GROUP BY 1 ORDER BY 2 DESC")

,coordinate_origin,count
0,tractor,16751
1,literature,14380
2,bcg,6


**Which systems are reported by the most papers?** (A cluster can have hundreds of *records*, one per arc or image in MasterLens, so we count distinct sources.)

In [6]:
sql("""
SELECT s.jname, count(DISTINCT sr.source) AS n_sources, count(*) AS n_records,
       string_agg(DISTINCT sr.source, ', ' ORDER BY sr.source) AS sources
FROM systems AS s JOIN system_records AS sr USING (system_id)
GROUP BY s.jname ORDER BY n_sources DESC, n_records DESC LIMIT 5
""")

,jname,n_sources,n_records,sources
0,J021737.15-051329.4,11,11,"Cabanac_et_al._2006, Canameras_et_al._2021, Ca..."
1,J090407.93-005953.0,10,10,"Cabanac_et_al._2006, Canameras_et_al._2021, Ca..."
2,J021411.22-040502.8,10,10,"Cabanac_et_al._2006, Canameras_et_al._2021, Ca..."
3,J222609.30+004141.9,9,10,"Canameras_et_al._2020, Canameras_et_al._2021, ..."
4,J022511.04-045433.5,9,9,"Canameras_et_al._2021, Cao_et_al._2015, Leaf_e..."


**Do the papers agree on the lens redshift?** The `raw` JSON lets us compare values that were never standardised:

In [7]:
sql("""
WITH z AS (
    SELECT sr.system_id, r.source,
           NULLIF(COALESCE(r.raw->>'Spec-z_L', r.raw->>'z_L'), '')::float AS z_l
    FROM system_records AS sr JOIN records AS r USING (source, record_id)
    WHERE sr.role = 'lens'
)
SELECT s.jname, count(*) AS n, round(min(z_l)::numeric, 4) AS z_min, round(max(z_l)::numeric, 4) AS z_max,
       string_agg(source || '=' || z_l, '; ') AS values
FROM z JOIN systems AS s USING (system_id)
WHERE z_l IS NOT NULL AND z_l = z_l
GROUP BY s.jname HAVING count(*) > 1 AND max(z_l) - min(z_l) > 0.05
ORDER BY max(z_l) - min(z_l) DESC LIMIT 10
""")

,jname,n,z_min,z_max,values
0,J101734.04-001227.2,3,0.4650,0.8457,Shu_et_al._2022=0.465; Wong_et_al._2022=0.8457...
1,J154402.71+261016.2,3,0.3359,0.7163,Talbot_et_al._2020=0.716284; Huang_et_al._2021...
2,J114311.27-013935.3,4,0.2880,0.6440,SuGOHI=0.644; Moustakas_et_al._2012=0.644; Hua...
3,J233113.19+003516.9,2,0.2800,0.6220,SuGOHI=0.28; Moustakas_et_al._2012=0.622
4,J023005.48-054059.3,2,0.1930,0.4980,Moustakas_et_al._2012=0.498; SuGOHI=0.193
5,J023158.61-031935.7,2,0.2740,0.5610,Moustakas_et_al._2012=0.561; SuGOHI=0.274
6,J204312.46-060953.8,2,0.5076,0.7926,Tran_et_al._2022=0.79261; Huang_et_al._2021=0....
7,J224836.36-012334.1,3,0.3980,0.6776,Huang_et_al._2021=0.677628; Moustakas_et_al._2...
8,J100018.49+023845.4,2,0.7100,0.9755,Faure_et_al._2008=0.71; Moustakas_et_al._2012=...
9,J193558.24+580908.9,2,0.3202,0.5774,Huang_et_al._2021=0.320223; Tran_et_al._2022=0...


**Which human decisions were recorded?** (The notes are currently written in Portuguese.)

In [8]:
sql("""
SELECT source, record_id, decision, left(note, 120) AS note
FROM manual_reviews ORDER BY updated_at DESC LIMIT 5
""")

,source,record_id,decision,note
0,Storfer_et_al._2022,1718,{'anchor': {'ls_id': '10995487905681820'}},Inspeção visual manual do painel name_coord_mi...
1,Moustakas_et_al._2012,5384,"{'position': {'ra': 147.84404166666667, 'dec':...",Literatura e NED: CASTLES FBQ0951+2635 em 09:5...
2,Moustakas_et_al._2012,2705,"{'position': {'ra': 45.62875, 'dec': 0.1005833...",Literatura e NED: CFRS 03.1077 G/G1 em 03:02:3...
3,Moustakas_et_al._2012,5236,"{'position': {'ra': 141.23279166666669, 'dec':...","Literatura: CASTLES, SDSS0924+0219 em 09:24:55..."
4,Moustakas_et_al._2012,4374,"{'position': {'ra': 116.218625, 'dec': 39.4575...","Literatura: Ebeling et al. 2007, ApJL 661, L33..."


## From the database to the files

Users will still receive flat tables: `load_postgres.py` exports `Database_v2.csv`, with **one line per record**, the system center and `JNAME`, and the old `JNAME_v1` for continuity. The database is the source of truth; the CSV files and the dashboard are views of it.

In [9]:
connection.close()